<a href="https://colab.research.google.com/github/AarushGoyal741/delhi-urban-expansion/blob/main/urban_expansion_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Urban Expansion Analysis of Delhi NCR (2001–2023)
### Using Random Forest Supervised Classification and Google Earth Engine

**Author:** Your Name  
**Programme:** India Space Academy – Winter Training on Remote Sensing, GIS and AI  
**Project Code:** P4  

---

## Objective
To quantify urban expansion in Delhi NCR between 2001 and 2023 using
supervised Random Forest classification on Landsat satellite imagery
processed via Google Earth Engine.

## Why Random Forest Instead of NDBI Thresholding?
Cross-sensor comparison between Landsat 5 (2001) and Landsat 8 (2023)
produces systematic radiometric offsets that cannot be fully corrected
by index-based thresholding. Supervised classification solves this by
training a separate model on each year's own imagery — making cross-sensor
differences irrelevant. Random Forest was chosen for its robustness to
noise, ability to handle multiple spectral bands simultaneously, and
strong performance in land cover classification literature.

In [ ]:
!pip install earthengine-api geemap -q

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os

os.makedirs('outputs', exist_ok=True)
print("✅ Packages loaded!")

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-aarushgoyal741')  # ← your project ID
print("✅ GEE initialized!")

In [ ]:
# Delhi NCR urban core AOI
aoi = ee.Geometry.Rectangle([76.95, 28.45, 77.45, 28.88])

Map = geemap.Map()
Map.centerObject(aoi, 11)
Map.addLayer(aoi, {'color': 'red'}, 'Delhi NCR AOI')
Map

In [ ]:
# ── Image preparation functions ───────────────────────────────────────
def harmonize_L5(image):
    scaled = image.select(
        ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']
    ).multiply(0.0000275).add(-0.2)
    slopes     = ee.Image.constant(
        [0.8474, 0.8483, 0.9047, 0.8462, 0.8937, 0.9071])
    intercepts = ee.Image.constant(
        [0.0003, 0.0088, 0.0061, 0.0412, 0.0254, 0.0172])
    harmonized = scaled.multiply(slopes).add(intercepts)
    return harmonized.rename(
        ['Blue','Green','Red','NIR','SWIR1','SWIR2']
    ).copyProperties(image, ['system:time_start'])

def scale_L8(image):
    scaled = image.select(
        ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
    ).multiply(0.0000275).add(-0.2)
    return scaled.rename(
        ['Blue','Green','Red','NIR','SWIR1','SWIR2']
    ).copyProperties(image, ['system:time_start'])

# ── Load imagery ──────────────────────────────────────────────────────
landsat_2001 = ee.ImageCollection("LANDSAT/LT05/C02/T1_L2") \
    .filterBounds(aoi) \
    .filterDate('2001-01-01', '2001-03-31') \
    .filter(ee.Filter.lt('CLOUD_COVER', 30)) \
    .map(harmonize_L5) \
    .median() \
    .clip(aoi)

landsat_2023 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2") \
    .filterBounds(aoi) \
    .filterDate('2023-01-01', '2023-03-31') \
    .filter(ee.Filter.lt('CLOUD_COVER', 15)) \
    .map(scale_L8) \
    .median() \
    .clip(aoi)

# Add spectral indices as extra features for the classifier
def add_indices(image):
    ndbi = image.normalizedDifference(['SWIR1','NIR']).rename('NDBI')
    ndvi = image.normalizedDifference(['NIR','Red']).rename('NDVI')
    mndwi = image.normalizedDifference(['Green','SWIR1']).rename('MNDWI')
    return image.addBands([ndbi, ndvi, mndwi])

image_2001 = add_indices(landsat_2001)
image_2023 = add_indices(landsat_2023)

# Bands used for classification
BANDS = ['Blue','Green','Red','NIR','SWIR1','SWIR2','NDBI','NDVI','MNDWI']

print("✅ Imagery prepared with spectral indices!")
print(f"   Classification bands: {BANDS}")

## Step 3: Define Training Samples

We define known land cover locations for 4 classes:
- **Class 0 — Urban**: Dense built-up (Connaught Place, roads, colonies)
- **Class 1 — Vegetation**: Parks, agricultural fields, green areas  
- **Class 2 — Bare Soil**: Open unpaved land, construction sites
- **Class 3 — Water**: Yamuna river, canals

These coordinates are chosen based on ground knowledge of Delhi —
locations that were definitively in each class in BOTH 2001 and 2023.
A separate Random Forest model is trained for each year independently,
so cross-sensor differences do not affect classification accuracy.

In [ ]:
# ── Training Samples — 3 classes only (Water handled by MNDWI mask) ──
# Removing water class from training entirely
# Water will be detected automatically using MNDWI index
# This is more accurate than point-based water training

# Class 0 = Urban
urban_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([77.2195, 28.6315]), {'landcover': 0}),  # Connaught Place
    ee.Feature(ee.Geometry.Point([77.1909, 28.6519]), {'landcover': 0}),  # Karol Bagh
    ee.Feature(ee.Geometry.Point([77.2373, 28.5677]), {'landcover': 0}),  # Lajpat Nagar
    ee.Feature(ee.Geometry.Point([77.0674, 28.7041]), {'landcover': 0}),  # Rohini
    ee.Feature(ee.Geometry.Point([77.2510, 28.5491]), {'landcover': 0}),  # Nehru Place
    ee.Feature(ee.Geometry.Point([77.0587, 28.5921]), {'landcover': 0}),  # Dwarka
    ee.Feature(ee.Geometry.Point([77.3261, 28.5700]), {'landcover': 0}),  # Noida sec 18
    ee.Feature(ee.Geometry.Point([77.0266, 28.4595]), {'landcover': 0}),  # Gurgaon
    ee.Feature(ee.Geometry.Point([77.2889, 28.6667]), {'landcover': 0}),  # Shahdara
    ee.Feature(ee.Geometry.Point([77.0833, 28.6289]), {'landcover': 0}),  # Janakpuri
    ee.Feature(ee.Geometry.Point([77.2167, 28.5245]), {'landcover': 0}),  # Saket
    ee.Feature(ee.Geometry.Point([77.1312, 28.7012]), {'landcover': 0}),  # Pitampura
])

# Class 1 = Vegetation
vegetation_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([77.1782, 28.5407]), {'landcover': 1}),  # Sanjay Van
    ee.Feature(ee.Geometry.Point([77.1689, 28.6401]), {'landcover': 1}),  # Central Ridge
    ee.Feature(ee.Geometry.Point([76.9823, 28.6102]), {'landcover': 1}),  # Najafgarh fields
    ee.Feature(ee.Geometry.Point([77.3801, 28.6734]), {'landcover': 1}),  # East agri fields
    ee.Feature(ee.Geometry.Point([77.2156, 28.5021]), {'landcover': 1}),  # Asola sanctuary
    ee.Feature(ee.Geometry.Point([77.3912, 28.5234]), {'landcover': 1}),  # Noida green belt
    ee.Feature(ee.Geometry.Point([77.1234, 28.8234]), {'landcover': 1}),  # North Delhi agri
    ee.Feature(ee.Geometry.Point([77.1834, 28.5134]), {'landcover': 1}),  # Mehrauli forest
])

# Class 2 = Bare Soil
bare_soil_points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([77.2934, 28.7134]), {'landcover': 2}),  # Yamuna floodplain
    ee.Feature(ee.Geometry.Point([77.0934, 28.8234]), {'landcover': 2}),  # Narela open land
    ee.Feature(ee.Geometry.Point([77.3534, 28.4534]), {'landcover': 2}),  # Faridabad outskirts
    ee.Feature(ee.Geometry.Point([76.9734, 28.7134]), {'landcover': 2}),  # West Delhi open
    ee.Feature(ee.Geometry.Point([77.0534, 28.8534]), {'landcover': 2}),  # North sandy land
    ee.Feature(ee.Geometry.Point([77.0134, 28.4534]), {'landcover': 2}),  # Gurgaon outskirts
])

# Merge — NO water class
training_points = urban_points \
    .merge(vegetation_points) \
    .merge(bare_soil_points)

total = training_points.size().getInfo()
print(f"✅ Training samples: {total} points")
print(f"   Urban: 12 | Vegetation: 8 | Bare Soil: 6")
print(f"   Water: auto-detected via MNDWI (more accurate)")

## Step 4: Train Random Forest Classifier

We train a separate Random Forest model for each year.
Each model uses 9 input features: 6 spectral bands + NDBI + NDVI + MNDWI.
100 decision trees are used for robust classification.

In [ ]:
def train_and_classify(image, training_points, bands, label):
    # Automatically mask water pixels using MNDWI
    # MNDWI > 0 = water with very high confidence
    mndwi = image.normalizedDifference(['Green','SWIR1'])
    water_mask = mndwi.lt(0.0)  # keep only non-water pixels

    # Apply water mask to image before classification
    image_masked = image.updateMask(water_mask)

    # Sample spectral values at training locations
    training_data = image_masked.select(bands).sampleRegions(
        collection=training_points,
        properties=['landcover'],
        scale=30
    )

    # Train Random Forest — 3 classes only (urban, veg, bare soil)
    classifier = ee.Classifier.smileRandomForest(
        numberOfTrees=100,
        seed=42
    ).train(
        features=training_data,
        classProperty='landcover',
        inputProperties=bands
    )

    # Classify non-water pixels
    classified = image_masked.select(bands).classify(classifier)

    # Add water back as class 3 using MNDWI mask
    water_layer = mndwi.gte(0.0).multiply(3)
    water_layer_masked = water_layer.updateMask(mndwi.gte(0.0))

    # Merge classified + water layer
    final = classified.blend(water_layer_masked).rename('classification')

    print(f"✅ {label} — classified with automatic water masking!")
    return final, classifier

print("Training 2001 classifier...")
classified_2001, clf_2001 = train_and_classify(
    image_2001, training_points, BANDS, "2001")

print("Training 2023 classifier...")
classified_2023, clf_2023 = train_and_classify(
    image_2023, training_points, BANDS, "2023")

# Visualize
class_vis = {
    'min': 0, 'max': 3,
    'palette': ['red', 'green', 'yellow', 'blue']
}
Map5 = geemap.Map()
Map5.centerObject(aoi, 11)
Map5.addLayer(classified_2001, class_vis, 'Classification 2001')
Map5.addLayer(classified_2023, class_vis, 'Classification 2023')
Map5

In [ ]:
# ── Extract urban class only (class = 0) ─────────────────────────────
urban_2001 = classified_2001.eq(0).rename('Urban_2001')
urban_2023 = classified_2023.eq(0).rename('Urban_2023')
new_urban  = urban_2023.subtract(urban_2001).gt(0).rename('New_Urban')

# ── Calculate areas ───────────────────────────────────────────────────
def calc_area_km2(mask, aoi, scale=30):
    area = mask.multiply(ee.Image.pixelArea()) \
               .reduceRegion(
                   reducer=ee.Reducer.sum(),
                   geometry=aoi,
                   scale=scale,
                   maxPixels=1e10
               )
    return round(list(area.getInfo().values())[0] / 1e6, 2)

area_2001  = calc_area_km2(urban_2001, aoi)
area_2023  = calc_area_km2(urban_2023, aoi)
area_new   = calc_area_km2(new_urban,  aoi)
pct_change = round(((area_2023 - area_2001) / area_2001) * 100, 1)
annual     = round(pct_change / 22, 2)

print("=" * 47)
print("      DELHI NCR URBAN EXPANSION STATS")
print("=" * 47)
print(f"  Method            : Random Forest (100 trees)")
print(f"  Features used     : {len(BANDS)} bands + indices")
print(f"  Built-up 2001     : {area_2001} km²")
print(f"  Built-up 2023     : {area_2023} km²")
print(f"  New Urban Added   : {area_new} km²")
print(f"  Total Change      : +{pct_change}%")
print(f"  Annual Growth     : +{annual}% per year")
print(f"  Study Period      : 22 years (2001–2023)")
print("=" * 47)

## Step 5: Visualize Results
Static publication-quality maps and charts for reporting.

In [ ]:
# ── Download arrays from GEE ──────────────────────────────────────────
def get_array(image, band, aoi, scale=150):
    arr = geemap.ee_to_numpy(
        image.select(band),
        region=aoi,
        scale=scale
    )
    return arr[:,:,0] if arr is not None else None

print("Downloading classification arrays...")
cls_arr_2001 = get_array(classified_2001, 'classification', aoi)
cls_arr_2023 = get_array(classified_2023, 'classification', aoi)

# Replace any masked/null values with -1
cls_arr_2001 = np.where(np.isnan(cls_arr_2001), -1, cls_arr_2001)
cls_arr_2023 = np.where(np.isnan(cls_arr_2023), -1, cls_arr_2023)

print(f"✅ Arrays downloaded — shape: {cls_arr_2001.shape}")
print(f"   Unique classes 2001: {np.unique(cls_arr_2001)}")
print(f"   Unique classes 2023: {np.unique(cls_arr_2023)}")

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# ── Figure 1: Side by side classification maps ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#1a1a2e')

# Color map: 0=Urban(red), 1=Vegetation(green),
#            2=Bare Soil(yellow), 3=Water(blue), -1=nodata(black)
cmap = mcolors.ListedColormap(['black','red','green','yellow','blue'])
bounds = [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

titles = ['Land Cover Classification — 2001\n(Landsat 5, Jan–Mar)',
          'Land Cover Classification — 2023\n(Landsat 8, Jan–Mar)']

for ax, arr, title in zip(axes,
                           [cls_arr_2001, cls_arr_2023],
                           titles):
    ax.set_facecolor('#1a1a2e')
    ax.imshow(arr, cmap=cmap, norm=norm)
    ax.set_title(title, color='white',
                 fontsize=13, fontweight='bold', pad=10)
    ax.axis('off')

legend_elements = [
    mpatches.Patch(color='red',    label='Urban / Built-up'),
    mpatches.Patch(color='green',  label='Vegetation'),
    mpatches.Patch(color='yellow', label='Bare Soil'),
    mpatches.Patch(color='blue',   label='Water'),
]
fig.legend(handles=legend_elements,
           loc='lower center', ncol=4,
           fontsize=11, framealpha=0.3,
           labelcolor='white',
           facecolor='#1a1a2e',
           edgecolor='white')

fig.suptitle('Delhi NCR Land Cover Classification (2001 vs 2023)\n'
             'Random Forest Supervised Classification — Google Earth Engine',
             color='white', fontsize=15, fontweight='bold')

plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.savefig('outputs/classification_maps.png',
            dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Classification maps saved!")

In [ ]:
# ── Figure 2: Urban change map ────────────────────────────────────────
urban_arr_2001 = (cls_arr_2001 == 0).astype(float)
urban_arr_2023 = (cls_arr_2023 == 0).astype(float)

# Change categories:
# 0 = never urban (both 0)
# 1 = existing urban (both 1)
# 2 = new urban (0→1)
# 3 = urban loss (1→0) — rare but possible
change = np.zeros_like(urban_arr_2001)
change[(urban_arr_2001==1) & (urban_arr_2023==1)] = 1  # existing
change[(urban_arr_2001==0) & (urban_arr_2023==1)] = 2  # new urban
change[(urban_arr_2001==1) & (urban_arr_2023==0)] = 3  # urban loss

fig, ax = plt.subplots(1, 1, figsize=(12, 10))
fig.patch.set_facecolor('#1a1a2e')
ax.set_facecolor('#0a0a1a')

cmap2 = mcolors.ListedColormap(['#1a1a2e','#FF8C00','#FF2020','#00BFFF'])
bounds2 = [-0.5, 0.5, 1.5, 2.5, 3.5]
norm2 = mcolors.BoundaryNorm(bounds2, cmap2.N)

ax.imshow(change, cmap=cmap2, norm=norm2)
ax.set_title('Urban Expansion Change Detection\nDelhi NCR 2001→2023',
             color='white', fontsize=15, fontweight='bold', pad=15)
ax.axis('off')

legend_elements2 = [
    mpatches.Patch(color='#1a1a2e', label='Non-urban',
                   edgecolor='gray'),
    mpatches.Patch(color='#FF8C00', label=f'Urban 2001 ({area_2001} km²)'),
    mpatches.Patch(color='#FF2020', label=f'New Urban 2001→2023 (+{area_new} km²)'),
    mpatches.Patch(color='#00BFFF', label='Urban loss (negligible)'),
]
ax.legend(handles=legend_elements2,
          loc='lower left', fontsize=11,
          framealpha=0.4, labelcolor='white',
          facecolor='#1a1a2e', edgecolor='white')

# Add stats annotation
stats_text = (f"Total Built-up 2001: {area_2001} km²\n"
              f"Total Built-up 2023: {area_2023} km²\n"
              f"Growth: +{pct_change}% in 22 years\n"
              f"Annual Rate: +{annual}% per year")
ax.text(0.98, 0.98, stats_text,
        transform=ax.transAxes,
        color='white', fontsize=11,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='#1a1a2e',
                  alpha=0.7, edgecolor='white'))

plt.tight_layout()
plt.savefig('outputs/urban_change_map.png',
            dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Urban change map saved!")

In [ ]:
# ── Figure 3: Statistics charts ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#1a1a2e')

for ax in axes:
    ax.set_facecolor('#0f3460')
    ax.spines[:].set_color('#555')
    ax.tick_params(colors='white', labelsize=11)
    ax.yaxis.label.set_color('white')
    ax.xaxis.label.set_color('white')
    ax.title.set_color('white')

# ── Plot 1: Bar chart — urban area comparison ─────────────────────────
bars = axes[0].bar(['2001', '2023'],
                   [area_2001, area_2023],
                   color=['#FF8C00','#FF2020'],
                   width=0.5, edgecolor='white', linewidth=0.8)
axes[0].set_title('Urban Area Comparison', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Area (km²)', fontsize=11)
axes[0].set_ylim(0, area_2023 * 1.3)
for bar, val in zip(bars, [area_2001, area_2023]):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 10,
                 f'{val} km²',
                 ha='center', color='white',
                 fontweight='bold', fontsize=11)

# ── Plot 2: Growth timeline ───────────────────────────────────────────
years_line  = [2001, 2023]
growth_line = [area_2001, area_2023]
axes[1].plot(years_line, growth_line,
             'o-', color='#FF6B6B',
             linewidth=2.5, markersize=10,
             markerfacecolor='white')
axes[1].fill_between(years_line, growth_line,
                     alpha=0.3, color='#FF6B6B')
axes[1].set_title('Urban Growth Timeline', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Built-up Area (km²)', fontsize=11)
axes[1].set_xlim(1999, 2025)
axes[1].set_ylim(0, area_2023 * 1.3)
for x, y in zip(years_line, growth_line):
    axes[1].annotate(f'{y} km²',
                     xy=(x, y), xytext=(0, 15),
                     textcoords='offset points',
                     ha='center', color='white',
                     fontweight='bold', fontsize=10)

# ── Plot 3: Land cover composition pie ────────────────────────────────
total_aoi_km2 = (77.45-76.95) * (28.88-28.45) * 111 * 111
non_urban_2023 = round(total_aoi_km2 - area_2023, 1)
sizes   = [area_2001,
           round(area_2023 - area_2001, 2),
           non_urban_2023]
labels  = [f'Urban 2001\n{area_2001} km²',
           f'New Urban\n{round(area_2023-area_2001,2)} km²',
           f'Non-urban\n{non_urban_2023} km²']
colors  = ['#FF8C00','#FF2020','#2d5a27']
explode = (0, 0.08, 0)

wedges, texts, autotexts = axes[2].pie(
    sizes, labels=labels, colors=colors,
    explode=explode, autopct='%1.1f%%',
    startangle=90,
    textprops={'color':'white','fontsize':10})
for at in autotexts:
    at.set_fontweight('bold')
axes[2].set_title('Urban Growth Composition 2023',
                  fontweight='bold', fontsize=13)

fig.suptitle(f'Delhi NCR Urban Expansion Statistics (2001–2023) | '
             f'+{pct_change}% growth | +{annual}% per year',
             color='white', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/stats_charts.png',
            dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Statistics charts saved!")

## Conclusion

This project quantified urban expansion in Delhi NCR over 22 years
using supervised Random Forest classification on harmonized Landsat
satellite imagery.

### Key Results

| Metric | Value |
|--------|-------|
| Built-up Area 2001 | 372.25 km² |
| Built-up Area 2023 | 535.68 km² |
| New Urban Area Added | 274.00 km² |
| **Total Growth** | **+43.9%** |
| Annual Growth Rate | +2.0% per year |
| Study Period | 22 years (2001–2023) |

### Methodology Summary
- **Satellite data**: Landsat 5 TM (2001) and Landsat 8 OLI (2023)
- **Harmonization**: Roy et al. (2016) coefficients applied to L5
- **Seasonal matching**: Jan–March dry season for both years
- **Classification**: Random Forest (100 trees, 9 features)
- **Features**: Blue, Green, Red, NIR, SWIR1, SWIR2, NDBI, NDVI, MNDWI
- **Training classes**: Urban, Vegetation, Bare Soil, Water

### Interpretation
Delhi NCR's built-up area grew by **43.9% over 22 years**, driven by
rapid population growth, satellite city development in Gurgaon, Noida
and Greater Noida, and planned residential expansion in Dwarka and Rohini.
The annual growth rate of 2.0% is consistent with published urbanization
studies for Delhi NCR.

### Limitations
- Training samples are point-based — polygon-based sampling would
  improve accuracy
- 30m spatial resolution means sub-pixel mixed land cover exists
- Cross-sensor harmonization corrects but does not fully eliminate
  inter-sensor differences
- Classification accuracy assessment (confusion matrix) would
  strengthen the analysis

### References
- Roy, D.P. et al. (2016). Characterization of Landsat-7 to Landsat-8
  reflective wavelength and normalized difference vegetation index
  continuity. *Remote Sensing of Environment*, 185, 57-70.
- Google Earth Engine: https://earthengine.google.com
- USGS Landsat Collection 2: https://www.usgs.gov/landsat-missions